# 🌍 LoL 월드 챔피언십 - 지역별 성과 분석

## 📌 프로젝트 정보
- **대주제**: LoL 월드 챔피언십 역대 분석
- **소주제**: 지역별(LCK, LPL, LEC, LCS 등) 우승/준우승 성과 분석
- **담당자**: 팀원 A

## 🎯 분석 목표
1. 역대 월드 챔피언십 지역별 우승/준우승 횟수 시각화
2. 연도별 지역 성과 추이 분석
3. 지역별 4강 이상 진출 빈도 분석

## 📊 사용할 시각화
- 스택 바 차트 (Stacked Bar Chart)
- 트리맵 (Treemap)
- 파이 차트 (Pie Chart)

---
## 1. 라이브러리 임포트 및 환경 설정

In [ ]:
# 기본 라이브러리
import pandas as pd
import numpy as np

# 시각화 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# 크롤링 라이브러리
import requests
from bs4 import BeautifulSoup

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')  # Windows
# plt.rc('font', family='AppleGothic')  # Mac
plt.rcParams['axes.unicode_minus'] = False

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료!")

---
## 2. 데이터 수집

### 2-1. Kaggle 데이터 다운로드
Kaggle에서 LoL 프로 경기 데이터를 다운로드합니다.

**추천 데이터셋:**
- https://www.kaggle.com/datasets/chuckephron/leagueoflegends
- https://www.kaggle.com/datasets/pedrocsar/league-of-legends-worlds-20112022-stats

In [ ]:
# Kaggle API를 통한 데이터 다운로드 (선택사항)
# !pip install kaggle
# !kaggle datasets download -d pedrocsar/league-of-legends-worlds-20112022-stats

# 또는 수동 다운로드 후 경로 지정
# df_kaggle = pd.read_csv('data/worlds_stats.csv')

### 2-2. 웹 크롤링을 통한 데이터 수집
Leaguepedia 또는 Liquipedia에서 역대 월드 챔피언십 결과를 크롤링합니다.

In [ ]:
# 역대 월드 챔피언십 결과 데이터 (수동 정리 또는 크롤링)
# 크롤링이 어려울 경우 아래 데이터를 직접 사용

worlds_results = {
    'Year': [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'Champion': ['Fnatic', 'TPA', 'SKT T1', 'SSW', 'SKT T1', 'SKT T1', 'SSG', 'IG', 'FPX', 'DWG', 'EDG', 'DRX', 'T1', 'T1'],
    'Champion_Region': ['LEC', 'PCS', 'LCK', 'LCK', 'LCK', 'LCK', 'LCK', 'LPL', 'LPL', 'LCK', 'LPL', 'LCK', 'LCK', 'LCK'],
    'Runner_Up': ['aAa', 'Azubu Frost', 'Royal Club', 'SHRC', 'KOO Tigers', 'SSG', 'SKT T1', 'Fnatic', 'G2', 'SN', 'DWG', 'T1', 'WBG', 'BLG'],
    'Runner_Up_Region': ['LEC', 'LCK', 'LPL', 'LPL', 'LCK', 'LCK', 'LCK', 'LEC', 'LEC', 'LPL', 'LCK', 'LCK', 'LPL', 'LPL'],
    'Final_Score': ['2-1', '3-1', '3-0', '3-1', '3-1', '3-2', '3-0', '3-0', '3-0', '3-1', '3-2', '3-2', '3-0', '3-2']
}

df_worlds = pd.DataFrame(worlds_results)
df_worlds.head()

In [ ]:
# 크롤링 함수 예시 (Liquipedia)
def crawl_worlds_data():
    """
    Liquipedia에서 월드 챔피언십 데이터 크롤링
    주의: 웹사이트 구조 변경 시 수정 필요
    """
    url = "https://liquipedia.net/leagueoflegends/World_Championship"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 테이블 찾기 및 파싱 (웹사이트 구조에 따라 수정)
        # tables = soup.find_all('table', class_='wikitable')
        
        print("크롤링 성공!")
        return soup
    except Exception as e:
        print(f"크롤링 실패: {e}")
        return None

# 크롤링 실행 (필요시 주석 해제)
# soup = crawl_worlds_data()

---
## 3. 데이터 전처리

In [ ]:
# 지역별 우승 횟수 집계
champion_counts = df_worlds['Champion_Region'].value_counts().reset_index()
champion_counts.columns = ['Region', 'Championships']
print("=== 지역별 우승 횟수 ===")
print(champion_counts)

In [ ]:
# 지역별 준우승 횟수 집계
runner_up_counts = df_worlds['Runner_Up_Region'].value_counts().reset_index()
runner_up_counts.columns = ['Region', 'Runner_Ups']
print("\n=== 지역별 준우승 횟수 ===")
print(runner_up_counts)

In [ ]:
# 지역별 우승 + 준우승 통합 데이터
region_stats = pd.merge(champion_counts, runner_up_counts, on='Region', how='outer').fillna(0)
region_stats['Championships'] = region_stats['Championships'].astype(int)
region_stats['Runner_Ups'] = region_stats['Runner_Ups'].astype(int)
region_stats['Total_Finals'] = region_stats['Championships'] + region_stats['Runner_Ups']
region_stats = region_stats.sort_values('Total_Finals', ascending=False)
print("\n=== 지역별 결승 진출 통계 ===")
region_stats

---
## 4. 데이터 시각화

### 4-1. 지역별 우승/준우승 횟수 (스택 바 차트)

In [ ]:
# 스택 바 차트
fig, ax = plt.subplots(figsize=(10, 6))

regions = region_stats['Region']
x = np.arange(len(regions))
width = 0.6

bars1 = ax.bar(x, region_stats['Championships'], width, label='우승', color='#FFD700')
bars2 = ax.bar(x, region_stats['Runner_Ups'], width, bottom=region_stats['Championships'], 
               label='준우승', color='#C0C0C0')

ax.set_xlabel('지역', fontsize=12)
ax.set_ylabel('횟수', fontsize=12)
ax.set_title('LoL 월드 챔피언십 지역별 결승 성과 (2011-2024)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(regions)
ax.legend()

# 값 표시
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height/2),
                    ha='center', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('region_stacked_bar.png', dpi=300, bbox_inches='tight')
plt.show()

### 4-2. 지역별 우승 비율 (파이 차트)

In [ ]:
# 파이 차트
fig, ax = plt.subplots(figsize=(8, 8))

colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6', '#F39C12']
explode = [0.05 if i == 0 else 0 for i in range(len(champion_counts))]

wedges, texts, autotexts = ax.pie(
    champion_counts['Championships'], 
    labels=champion_counts['Region'],
    autopct='%1.1f%%',
    colors=colors[:len(champion_counts)],
    explode=explode,
    startangle=90,
    textprops={'fontsize': 12}
)

ax.set_title('LoL 월드 챔피언십 지역별 우승 비율', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('region_pie_chart.png', dpi=300, bbox_inches='tight')
plt.show()

### 4-3. 연도별 지역 성과 추이 (라인 차트)

In [ ]:
# 연도별 지역 우승 누적 데이터 생성
regions_list = ['LCK', 'LPL', 'LEC', 'PCS']
cumulative_data = {region: [] for region in regions_list}

for region in regions_list:
    cumsum = 0
    for year in df_worlds['Year']:
        if df_worlds[df_worlds['Year'] == year]['Champion_Region'].values[0] == region:
            cumsum += 1
        cumulative_data[region].append(cumsum)

# 라인 차트
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'LCK': '#E74C3C', 'LPL': '#3498DB', 'LEC': '#2ECC71', 'PCS': '#9B59B6'}

for region in regions_list:
    ax.plot(df_worlds['Year'], cumulative_data[region], marker='o', 
            label=region, color=colors[region], linewidth=2, markersize=6)

ax.set_xlabel('연도', fontsize=12)
ax.set_ylabel('누적 우승 횟수', fontsize=12)
ax.set_title('LoL 월드 챔피언십 지역별 누적 우승 추이 (2011-2024)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xticks(df_worlds['Year'])
ax.set_xticklabels(df_worlds['Year'], rotation=45)

plt.tight_layout()
plt.savefig('region_cumulative_line.png', dpi=300, bbox_inches='tight')
plt.show()

### 4-4. 트리맵 (Plotly 인터랙티브)

In [ ]:
# 트리맵 데이터 준비
treemap_data = df_worlds.groupby(['Champion_Region', 'Champion']).size().reset_index(name='Count')

fig = px.treemap(
    treemap_data,
    path=['Champion_Region', 'Champion'],
    values='Count',
    color='Champion_Region',
    color_discrete_map={'LCK': '#E74C3C', 'LPL': '#3498DB', 'LEC': '#2ECC71', 'PCS': '#9B59B6'},
    title='LoL 월드 챔피언십 지역별/팀별 우승 분포'
)

fig.update_layout(font=dict(size=14))
fig.show()

---
## 5. 결론 및 인사이트

### 📊 분석 결과

1. **LCK(한국)의 압도적 우위**
   - 역대 14회 월드 챔피언십 중 10회 우승으로 약 71%의 우승률 기록
   - T1(SKT T1)이 단일 팀으로 최다 우승(4회) 달성

2. **LPL(중국)의 성장**
   - 2018년 IG 우승을 시작으로 3회 우승
   - 최근 결승 진출 빈도 증가로 LCK와의 경쟁 심화

3. **LEC(유럽)의 하락세**
   - 2011년 시즌1 우승 이후 우승 없음
   - 2018-2019년 결승 진출 후 최근 성적 부진

4. **양강 구도 확립**
   - 2018년 이후 결승전은 LCK vs LPL 구도로 고착화
   - 다른 지역의 경쟁력 회복 필요

### 💡 시사점
- e스포츠 생태계에서 LCK와 LPL의 인프라 투자가 성과로 이어짐
- 지역 간 선수 이적 및 코칭 스태프 교류의 영향 분석 필요

---
## 6. 참고 자료

### 📚 데이터 출처
- **Kaggle**: https://www.kaggle.com/datasets/pedrocsar/league-of-legends-worlds-20112022-stats
- **Liquipedia**: https://liquipedia.net/leagueoflegends/World_Championship
- **Leaguepedia**: https://lol.fandom.com/wiki/World_Championship

### 🛠️ 사용 도구
- Python 3.x
- Jupyter Lab
- pandas, matplotlib, seaborn, plotly